In [ ]:
# load the posts: one row per post, with the verdict from its flair
import pandas as pd

posts = pd.read_csv("https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data/week10_posts.csv")
posts.shape  # (785, 4)

In [ ]:
# the three example posts from the slides
examples = [69, 773, 175]
posts.loc[examples, ["title", "verdict"]]

## TF-IDF

In [ ]:
# make the TF-IDF table: one row per post, one column per word
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english", min_df=30)  # words in at least 30 posts
weights = pd.DataFrame(tfidf.fit_transform(posts["text"]).toarray(),
                       columns=tfidf.get_feature_names_out())
weights.shape  # (785, 350)

In [ ]:
# the weights of six words for the three example posts
weights.loc[examples, ["kid", "refused", "parents", "school", "daughter", "honestly"]].round(2)

In [ ]:
# average the TF-IDF weights of the NTA posts, and of the YTA posts
by_verdict = pd.DataFrame({
    "NTA": weights[posts["verdict"] == "NTA"].mean(),
    "YTA": weights[posts["verdict"] == "YTA"].mean(),
})

# divide the YTA average by the NTA average: above 1 means the word is more YTA
by_verdict["ratio"] = by_verdict["YTA"] / by_verdict["NTA"]
by_verdict.sort_values("ratio")

In [ ]:
# plot the 6 most NTA-like and the 6 most YTA-like words
by_ratio = by_verdict.sort_values("ratio")
selected = pd.concat([by_ratio.head(6), by_ratio.tail(6)])
selected[["NTA", "YTA"]].plot.barh(figsize=(6, 5), xlabel="average TF-IDF weight", ylabel="")

## Embeddings

In [ ]:
# turn every post into 384 numbers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # downloads about 90 MB the first time
embeddings = model.encode(posts["text"].tolist(), normalize_embeddings=True)
embeddings.shape  # (785, 384)

In [ ]:
# the first 4 of the 384 numbers for the three example posts
pd.DataFrame(embeddings[examples, :4], index=examples).round(2)

In [ ]:
# find the posts closest to a sentence
query = model.encode(["My roommate keeps eating my food."], normalize_embeddings=True)[0]
posts["similarity"] = embeddings @ query  # how close each post is to the sentence
posts.sort_values("similarity", ascending=False).head(4)[["similarity", "verdict", "title"]]

## An LLM label: does the poster admit fault?

Needs `GEMINI_API_KEY` in your `.env` file. 15 calls.

In [ ]:
# connect to gemini with the key from your .env file
import os

from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [ ]:
# ask gemini whether each example poster admits fault; the definition is the system instruction
from google.genai import types

definition = ("Label whether the poster admits that they did something wrong.\n"
              "yes: the poster says that something they did was wrong, unfair,\n"
              "or an overreaction, even in passing.\n"
              "no: the poster does not admit any fault. Give a reason of one sentence.")

def label(text):
    response = client.models.generate_content(model="gemini-3.5-flash-lite", contents=text,
        config=types.GenerateContentConfig(system_instruction=definition))
    return response.text

posts.loc[examples, "text"].map(label)  # 3 posts, 3 calls

In [ ]:
# the same question, but gemini must answer with a label (yes or no) and a reason
import json

config = types.GenerateContentConfig(
    system_instruction=definition,
    response_mime_type="application/json",  # answer in JSON
    response_schema={                       # with exactly these two fields
        "type": "object",
        "properties": {
            "label": {"type": "string", "enum": ["yes", "no"]},  # only yes or no allowed
            "reason": {"type": "string"},
        },
        "required": ["label", "reason"],
    },
)

def label_json(text):
    response = client.models.generate_content(model="gemini-3.5-flash-lite", contents=text, config=config)
    return json.loads(response.text)  # turn the JSON text into a dictionary

answers = [label_json(text) for text in posts.loc[examples, "text"]]  # 3 calls
pd.DataFrame(answers, index=examples)

In [ ]:
# ask about three borderline posts three times each, with the schema
borderline = posts.index[posts["id"].isin(["t3_r956gj", "t3_ep1uwm", "t3_gxzgtl"])]
runs = pd.DataFrame(index=borderline)
for run in ["run 1", "run 2", "run 3"]:
    runs[run] = [label_json(text)["label"] for text in posts.loc[borderline, "text"]]
runs["same every time"] = runs.nunique(axis=1) == 1
runs  # 9 calls

## Agreement between two labelers

In [ ]:
# agreement and kappa for two labelers on 10 posts
from sklearn.metrics import cohen_kappa_score

labels_a = ["yes", "yes", "no", "no", "no", "no", "no", "no", "no", "no"]
labels_b = ["yes", "no", "yes", "no", "no", "no", "no", "no", "no", "no"]
agreement = (pd.Series(labels_a) == pd.Series(labels_b)).mean()
agreement, cohen_kappa_score(labels_a, labels_b)  # (0.8, 0.375)